# Réseau de neurones simple avec PyTorch

Dans ce TP, nous allons construire deux réseaux de neurones avec PyTorch, en écrivant nous-mêmes leurs paramètres, leur perte et leur boucle d'entraînement : un réseau sans couche cachée d'abord, puis un perceptron multicouche. Le TP suivant construira les mêmes réseaux avec PyTorch Lightning, qui écrit la boucle à notre place.

Les données sont celles de MNIST, le jeu *Hello World!* du machine learning : des images de 28 $\times$ 28 pixels en niveaux de gris, qui représentent des chiffres de 0 à 9. Chaque image est associée à un label, le chiffre que l'image est censée représenter.

Les étapes :

1. Prendre en main les tenseurs PyTorch
2. Explorer et préparer les données, mettre de côté un ensemble de validation
3. Écrire un réseau sans couche cachée, sa perte, son exactitude et son `DataLoader`
4. L'entraîner par descente de gradient écrite à la main, puis avec un optimiseur PyTorch
5. Compiler le modèle avec `torch.compile`, et mesurer ce que la compilation coûte et rapporte
6. Passer à un perceptron multicouche et le régulariser
7. Suivre ses entraînements dans TensorBoard, choisir son learning rate sur la validation
8. L'évaluer sur l'ensemble de test, le sauvegarder et le recharger

Ce notebook n'a pas besoin de GPU : chaque entraînement dure de quelques secondes à quelques minutes sur le processeur de Colab.

## Imports

In [ ]:
import time
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import seaborn
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

# Les calculs auront lieu sur le GPU s'il y en a un
device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch fournit les tenseurs, les gradients, les optimiseurs et le chargement des données par batchs ; torchvision, le jeu MNIST ; `torch.utils.tensorboard`, les journaux d'entraînement qu'affiche TensorBoard. [`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) et [`seaborn`](https://seaborn.pydata.org/) servent aux graphiques et aux opérations simples sur les tableaux.

## Prise en main des tenseurs

Sur Colab, PyTorch est déjà installé. Sur votre machine, installez-le avec `pip install torch torchvision`. Sous Linux sans GPU NVIDIA, ajoutez `--index-url https://download.pytorch.org/whl/cpu` : sans lui, pip télécharge la version CUDA et plusieurs Go de bibliothèques NVIDIA inutiles (slide « Installer PyTorch »). Pour un autre système ou un autre GPU, [pytorch.org](https://pytorch.org/get-started/locally/) donne la commande adaptée.

Avant de construire des réseaux, manipulons les tenseurs PyTorch, dont l'API est très proche de celle de NumPy (slides « NumPy et PyTorch » et « Calculer sur GPU »).

1. Créez un tenseur `a` de forme (2, 3) contenant les entiers de 0 à 5 (`torch.arange`, puis `reshape`). Quel est son `dtype` ? Convertissez-le en flottants (`.float()`), puis créez un tenseur `b` de forme (3,) rempli de 1 (`torch.ones`).
2. Calculez `a + b` et le produit matriciel `a @ b`. Quelles sont les formes des résultats ? Pourquoi `a + torch.ones(2)` échoue-t-il ? Une opération qui échoue lève une `RuntimeError` : attrapez-la avec `try`/`except` pour afficher son message.
3. Extrayez la deuxième colonne de `a`, puis sa dernière ligne. Ajoutez une dimension de taille 1 en tête de `a` avec `unsqueeze` : quelle forme obtient-on ? À quoi sert une telle dimension avec un modèle PyTorch ?
4. Convertissez `a` en tableau NumPy (`.numpy()`), puis de nouveau en tenseur (`torch.from_numpy`). Copiez `a` sur `device` (défini dans la cellule d'imports) et affichez son attribut `device`.

**Documentation utile**

- [`torch.arange`](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)
- [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- [`torch.Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unsqueeze.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.arange(6).reshape(2, 3)`, puis `a.float()`
- `a[:, 1]` : la deuxième colonne ; `a[-1]` : la dernière ligne
- `a.unsqueeze(0)` : une dimension de taille 1 en position 0
- `a.to(device)` : une copie sur `device` ; son attribut `device` dit où elle se trouve
- Une opération qui échoue, dans un `try` :

```python
try:
  a + torch.ones(2)
except RuntimeError as erreur:
  print(erreur)
```

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
a = torch.arange(...).reshape(...)
print(a.dtype)
a = a.float()
b = ...

print((a + b).shape, (a @ b).shape)
try:
  ...  # a + torch.ones(2)
except RuntimeError as erreur:
  print(erreur)

print(a[:, 1], ...)  # deuxième colonne, dernière ligne
print(a.unsqueeze(...).shape)

print(type(a.numpy()), torch.from_numpy(...).dtype)
print(a.to(...).device)
```

### Solution

In [ ]:
a = torch.arange(6).reshape(2, 3)
print(a, a.dtype)
a = a.float()
b = torch.ones(3)

# b est diffusé (broadcasting) sur chaque ligne de a
print(a + b, (a + b).shape)
# Produit matrice-vecteur : la somme de chaque ligne de a
print(a @ b, (a @ b).shape)
try:
  a + torch.ones(2)
except RuntimeError as erreur:
  print("a + torch.ones(2) :", erreur)

print(a[:, 1])  # deuxième colonne
print(a[-1])  # dernière ligne
print(a.unsqueeze(0).shape)  # dimension de batch de taille 1

a_numpy = a.numpy()
print(type(a_numpy), torch.from_numpy(a_numpy).dtype)
print(a.to(device).device)

- `torch.arange` crée des entiers 64 bits (`torch.int64`) ; les réseaux de neurones calculent en flottants, par défaut `torch.float32`.
- `a + b` a la forme (2, 3) et `a @ b` la forme (2,). Pour le broadcasting, les dimensions sont comparées en partant de la dernière : elles doivent être égales, ou l'une d'elles doit valoir 1 ou être absente. (2, 3) et (3,) sont compatibles, mais pas (2, 3) et (2,) : il faudrait la forme (2, 1), `torch.ones(2).unsqueeze(1)`.
- `a.unsqueeze(0)` a la forme (1, 2, 3). Les modèles PyTorch traitent des batchs, la première dimension étant celle des exemples : pour prédire sur un seul exemple, on lui ajoute une dimension de batch de taille 1.
- `a.numpy()` partage la mémoire du tenseur, sans copie : il faut pour cela un tenseur sur CPU (`.cpu()`) et hors du graphe d'autograd (`.detach()`). `torch.from_numpy` conserve le type NumPy (`float32` ici, mais `float64` pour un tableau NumPy créé par défaut).
- Sans GPU, `a.to(device).device` vaut `cpu` ; avec un GPU, `cuda:0`.

## Récupération des données

In [ ]:
donnees_entrainement = torchvision.datasets.MNIST("donnees", train=True, download=True)
donnees_test = torchvision.datasets.MNIST("donnees", train=False, download=True)
X_train, y_train = donnees_entrainement.data, donnees_entrainement.targets
X_test, y_test = donnees_test.data, donnees_test.targets

## Regardons les données

Utilisez la cellule suivante pour explorer les tenseurs `X_train`, `y_train`, `X_test` et `y_test` : leur forme (`shape`), leur type (`dtype`), leurs valeurs extrêmes, et quelques images avec `plt.imshow`.

Vous essayerez de répondre aux questions suivantes :

- Combien y a-t-il d'images au total dans ce dataset ?
- Sous quelle forme sont stockées les images ? les labels ?
- Les classes sont-elles équilibrées ?

**Documentation utile**

- [`torch.bincount`](https://docs.pytorch.org/docs/stable/generated/torch.bincount.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()`, `X_train.max()` : forme, type et valeurs extrêmes
- `torch.bincount(y_train)` : le nombre d'exemples de chaque classe, de 0 à 9
- `plt.imshow(X_train[0], cmap="gray_r")` : la première image, en noir sur blanc

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, ...)
print(X_test.shape, ...)

print(torch.bincount(...))

plt.imshow(..., cmap="gray_r")
plt.title(f"Label : {int(y_train[0])}")
plt.show()
```

### Solution

In [ ]:
print(f"Images d'entraînement : {tuple(X_train.shape)}, {X_train.dtype}")
print(f"Valeurs des pixels : de {int(X_train.min())} à {int(X_train.max())}")
print(f"Labels d'entraînement : {tuple(y_train.shape)}, {y_train.dtype}")
print(f"Images et labels de test : {tuple(X_test.shape)}, {tuple(y_test.shape)}")
print(f"Exemples par classe : {torch.bincount(y_train).tolist()}")

plt.imshow(X_train[0], cmap="gray_r")
plt.title(f"Premier exemple du dataset ({int(y_train[0])})")
plt.show()

seaborn.countplot(x=y_train.numpy())
plt.title("Décompte des différentes classes (chiffres)")
plt.ylabel("Décompte")
plt.xlabel("Chiffre")
plt.show()

- 70 000 images : 60 000 d'entraînement et 10 000 de test.
- Les images sont un tenseur `torch.uint8` de forme (60 000, 28, 28) : un entier de 0 (fond) à 255 (encre) par pixel. Les labels sont un tenseur `torch.int64` de forme (60 000,), un chiffre de 0 à 9 par image.
- Les classes sont à peu près équilibrées : de 5 421 exemples (le 5) à 6 742 (le 1). L'exactitude, la proportion de bonnes réponses, est donc une métrique pertinente.

## Image moyenne de chaque chiffre

*Calculez l'image moyenne de chaque chiffre : pour chaque label `k` de 0 à 9, la moyenne, pixel par pixel, des images d'entraînement de label `k`. Affichez les 10 images moyennes côte à côte, chacune avec son chiffre en titre. Que montrent-elles ?*

**Documentation utile**

- [`torch.mean`](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Une ligne de 10 sous-graphiques, un par chiffre
f, ax = plt.subplots(1, 10, figsize=(15, 2))

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `y_train == k` : un masque, un tenseur de booléens qui vaut `True` pour les exemples de label `k`
- `X_train[masque]` : les seules images pour lesquelles le masque vaut `True`, un tenseur de forme (nombre d'images de label `k`, 28, 28)
- `.float()` : les pixels sont des entiers `torch.uint8`, dont PyTorch refuse de calculer la moyenne
- `.mean(dim=0)` : la moyenne sur la dimension 0, celle des exemples ; il reste une image de forme (28, 28)
- `ax[k].imshow(image, cmap="gray_r")`, `ax[k].set_title(...)` et `ax[k].axis("off")` : l'image, son titre et sans axes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
f, ax = plt.subplots(1, 10, figsize=(15, 2))
for k in range(10):
  image_moyenne = X_train[...].float().mean(dim=...)
  ax[k].imshow(..., cmap="gray_r")
  ax[k].set_title(...)
  ax[k].axis("off")
```

### Solution

In [ ]:
f, ax = plt.subplots(1, 10, figsize=(15, 2))

for k in range(10):
  # Le masque y_train == k garde les images de label k ; leur moyenne sur la
  # dimension 0, celle des exemples, est une image de 28 x 28 pixels
  image_moyenne = X_train[y_train == k].float().mean(dim=0)
  ax[k].imshow(image_moyenne, cmap="gray_r")
  ax[k].set_title(str(k))
  ax[k].axis("off")

Chaque image moyenne montre la forme typique de son chiffre : foncée là où presque toutes les écritures passent, floue là où elles varient d'une personne à l'autre. Le 1, un trait plus ou moins penché, donne l'image la plus étroite.

Le réseau sans couche cachée de la suite fait un calcul proche : il compare chaque image à un gabarit par chiffre, une colonne de sa matrice de poids, et choisit le chiffre dont le gabarit lui ressemble le plus.

## Transformation des données

Trois transformations préparent les données pour le réseau :

1. Mettre les images à plat : un réseau sans couche cachée prend un vecteur par exemple, de forme `(_, 28²)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt qu'entre $0$ et $255$.
3. Copier les images et les labels sur `device`, où le modèle calculera (slide « Calculer sur GPU »).

*Appliquez ces trois transformations à `X_train` et à `X_test`, et la troisième à `y_train` et à `y_test`.*

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X.reshape(-1, 28 * 28)` : `-1` laisse PyTorch calculer le nombre de lignes
- `X / 255` : des valeurs entre 0 et 1 ; la division d'un tenseur d'entiers donne des `torch.float32`
- `X.to(device)` : la copie sur `device`

Attendu : `X_train.shape` vaut `torch.Size([60000, 784])`, `X_train.dtype` vaut `torch.float32` et `X_train.max()` vaut 1.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = (X_train.reshape(-1, ...) / ...).to(device)
X_test = ...
y_train, y_test = y_train.to(...), ...
print(X_train.shape, X_train.dtype, X_train.max())
```

### Solution

In [ ]:
dim_entree = 28 * 28

# Mise à plat : un vecteur de dim_entree pixels par image, puis passage dans
# [0, 1] et copie sur le périphérique de calcul
X_train = (X_train.reshape(-1, dim_entree) / 255).to(device)
X_test = (X_test.reshape(-1, dim_entree) / 255).to(device)
y_train, y_test = y_train.to(device), y_test.to(device)
print(X_train.shape, X_train.dtype, X_train.max())

## Séparation entraînement / validation

Pour choisir les hyper-paramètres (learning rate, taille du réseau…), on compare des modèles sur un ensemble de **validation**. L'ensemble de test est réservé à l'évaluation finale : s'en servir pour faire des choix rendrait cette évaluation trop optimiste.

On met de côté les 10 000 derniers exemples d'entraînement pour la validation :

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Réseau de neurones sans couche cachée

Un réseau de neurones sans couche cachée calcule $\text{softmax}(XW + b)$ : c'est la **régression logistique multinomiale**. `W` est la matrice des poids, multipliée à l'entrée, et `b` le vecteur des biais, ajouté ensuite. $XW + b$ donne 10 scores par exemple, les **logits**, que la softmax transforme en probabilités.

En PyTorch, le modèle s'arrête aux logits, sans softmax finale : la fonction de perte calcule elle-même le logarithme de la softmax (slide « Piège classique : softmax et entropie croisée »).

### Initialisation des paramètres

*Complétez la fonction `cree_poids_reseau_simple(nb_entrees, nb_sorties)`, qui renvoie deux [paramètres PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) créés sur `device` : `W`, de forme `(nb_entrees, nb_sorties)`, tiré d'une loi normale (`torch.randn`) et divisé par $\sqrt{n_\text{entrées}}$ ; `b`, un vecteur de zéros de forme `(nb_sorties,)`.*

Cette division donne aux poids une variance de $1 / n_\text{entrées}$ : la sortie de la couche, une somme de $n_\text{entrées}$ termes, reste de l'ordre de grandeur de ses entrées, quel que soit leur nombre (slide « Initialisation des poids »).

**Documentation utile**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.zeros`](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

In [ ]:
def cree_poids_reseau_simple(
  nb_entrees: int, nb_sorties: int
) -> tuple[nn.Parameter, nn.Parameter]:
  pass  # Votre code ici

#### Conseils supplémentaires

- `torch.randn(nb_entrees, nb_sorties, device=device)` : un tenseur de moyenne 0 et d'écart type 1, créé directement sur `device`
- `nb_entrees**0.5` : la racine carrée de `nb_entrees`
- `torch.zeros(nb_sorties, device=device)` : les biais, nuls au départ
- `nn.Parameter(tenseur)` : un tenseur dont PyTorch calcule le gradient, et qu'un `nn.Module` reconnaît comme paramètre

Attendu : `W` a la forme (784, 10) et un écart type (`W.detach().std()`) proche de $1 / 28 \approx 0{,}036$.

```python
def cree_poids_reseau_simple(
  nb_entrees: int, nb_sorties: int
) -> tuple[nn.Parameter, nn.Parameter]:
  W = torch.randn(..., ..., device=device) / ...
  b = torch.zeros(..., device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = cree_poids_reseau_simple(784, 10)
print(W.shape, b.shape, W.detach().std())
```

#### Solution

In [ ]:
def cree_poids_reseau_simple(
  nb_entrees: int, nb_sorties: int
) -> tuple[nn.Parameter, nn.Parameter]:
  # Variance 1 / nb_entrees : des sorties de l'ordre de 1, quel que soit nb_entrees
  W = torch.randn(nb_entrees, nb_sorties, device=device) / nb_entrees**0.5
  b = torch.zeros(nb_sorties, device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = cree_poids_reseau_simple(784, 10)
print(W.shape, b.shape, W.detach().std())

### Définition du modèle

*Complétez la méthode `forward` de `ReseauSimple`, qui renvoie les logits $XW + b$ de chaque exemple.*

In [ ]:
class ReseauSimple(nn.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    self.W, self.b = cree_poids_reseau_simple(nb_entrees, nb_sorties)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Votre code ici

#### Conseils supplémentaires

- `X @ self.W + self.b` : le produit matriciel, puis l'ajout des biais, diffusé sur chaque ligne

Attendu : `ReseauSimple()(X_train[:5])` a la forme (5, 10), et la softmax de chaque ligne (`torch.softmax(..., dim=-1)`) a une somme de 1.

```python
class ReseauSimple(nn.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    self.W, self.b = cree_poids_reseau_simple(nb_entrees, nb_sorties)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    return ... @ ... + ...
```

#### Solution

In [ ]:
class ReseauSimple(nn.Module):
  def __init__(self, nb_entrees: int = 28**2, nb_sorties: int = 10) -> None:
    super().__init__()
    self.W, self.b = cree_poids_reseau_simple(nb_entrees, nb_sorties)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Les logits : un score par classe, sans softmax
    return X @ self.W + self.b


logits = ReseauSimple()(X_train[:5])
# La softmax en fait des probabilités, de somme 1 sur chaque ligne
print(logits.shape, torch.softmax(logits, dim=-1).sum(dim=-1))

### Calcul de la fonction de perte

La fonction de perte standard en classification multi-classes est l'entropie croisée : la moyenne, sur les exemples, de l'opposé de la log-probabilité que le modèle donne à la bonne classe. Les log-probabilités se calculent depuis les logits avec `torch.log_softmax` : le logarithme de la softmax, calculé en une fois, est plus stable numériquement que la softmax suivie d'un logarithme.

*Complétez la fonction `entropie_croisee_categorielle`, qui prend les logits du modèle et les labels entiers `y` (de 0 à 9), dans l'ordre de `nn.functional.cross_entropy`, en écrivant le calcul vous-même : les log-probabilités, celle de la bonne classe de chaque exemple, puis l'opposé de leur moyenne. Comparez votre résultat à celui de `nn.functional.cross_entropy` (slide « Piège classique : softmax et entropie croisée »). Quelle valeur prend la perte pour un modèle qui n'a pas encore appris ?*

**Documentation utile**

- [`torch.log_softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)

In [ ]:
def entropie_croisee_categorielle(
  logits: torch.Tensor, y: torch.Tensor
) -> torch.Tensor:
  pass  # Votre code ici

#### Conseils supplémentaires

- `torch.log_softmax(logits, dim=-1)` : les log-probabilités, une ligne par exemple, une colonne par classe
- `log_probas[torch.arange(len(y), device=y.device), y]` : indexé par deux tenseurs d'entiers, `log_probas` donne, pour chaque ligne `i`, sa colonne `y[i]` : la log-probabilité de la bonne classe de chaque exemple (`device=y.device` crée les numéros de ligne là où sont les labels)
- `.mean()`, précédé d'un signe moins : l'opposé de la moyenne
- `nn.functional.cross_entropy(logits, y)` fait ce calcul en une fois, `log_softmax` compris : c'est elle qu'on utilise en pratique. Ses arguments viennent dans le même ordre, les logits d'abord.

Un modèle qui n'a pas appris donne à peu près la même probabilité, 1/10, à chaque classe : sa perte vaut environ $-\log(1/10) = \log 10 \approx 2{,}3$, et les deux fonctions donnent la même valeur.

```python
def entropie_croisee_categorielle(
  logits: torch.Tensor, y: torch.Tensor
) -> torch.Tensor:
  log_probas = torch.log_softmax(..., dim=...)
  return -log_probas[torch.arange(len(y), device=y.device), ...].mean()


logits = ReseauSimple()(X_train)
print(entropie_croisee_categorielle(logits, y_train))
print(nn.functional.cross_entropy(..., ...))
```

#### Solution

In [ ]:
def entropie_croisee_categorielle(
  logits: torch.Tensor, y: torch.Tensor
) -> torch.Tensor:
  log_probas = torch.log_softmax(logits, dim=-1)
  # Pour chaque exemple i, la log-probabilité de sa bonne classe y[i]
  log_probas_bonne_classe = log_probas[torch.arange(len(y), device=y.device), y]
  return -log_probas_bonne_classe.mean()


# Environ log(10) = 2,3 : le modèle répond presque au hasard
logits = ReseauSimple()(X_train)
print(entropie_croisee_categorielle(logits, y_train))
# La même valeur : cross_entropy applique log_softmax elle-même
print(nn.functional.cross_entropy(logits, y_train))

### Métriques

Pour savoir si un modèle apprend, on mesure ses performances. Nous utiliserons la plus simple, mais aussi une des plus informatives : l'exactitude (*accuracy*), le nombre de bonnes prédictions sur le nombre total de prédictions.

*Complétez la fonction `exactitude`. La classe prédite est celle du plus grand logit (`torch.argmax`), la softmax ne changeant pas l'ordre des scores : comparez-la aux labels, puis calculez la proportion de bonnes réponses.*

**Documentation utile**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
def exactitude(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  pass  # Votre code ici

#### Conseils supplémentaires

- `torch.argmax(y_pred, dim=-1)` : la classe prédite pour chaque exemple
- `predictions == y` : un tenseur de booléens
- `.float().mean()` : la proportion de `True`
- `.item()` : un nombre Python, plus simple à afficher et à stocker

Attendu : environ 0,1 pour un modèle qui n'a pas appris, une chance sur dix.

```python
def exactitude(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(..., dim=...)
  return (... == ...).float().mean().item()


print(exactitude(y_val, ReseauSimple()(X_val)))
```

#### Solution

In [ ]:
def exactitude(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(y_pred, dim=-1)
  return (predictions == y).float().mean().item()


print(exactitude(y_val, ReseauSimple()(X_val)))

## Création d'un `DataLoader` PyTorch

Le modèle apprendra par mini-batchs : à chaque pas, un petit groupe d'exemples tirés au hasard. Un `DataLoader` les prépare (slides « Charger ses données : `DataLoader` » et « Boucle d'entraînement complète »).

*Complétez la fonction `fabrique_dataset`, qui enveloppe les tenseurs `X` et `y` dans un `TensorDataset`, puis dans un `DataLoader` qui renvoie des batchs `(X_lot, y_lot)` de `taille_lot` exemples, mélangés à chaque epoch.*

In [ ]:
def fabrique_dataset(
  X: torch.Tensor, y: torch.Tensor, taille_lot: int = 128
) -> DataLoader:
  pass  # Votre code ici

### Conseils supplémentaires

- `TensorDataset(X, y)` : un dataset dont l'exemple `i` est le couple `(X[i], y[i])`
- `DataLoader(dataset, batch_size=taille_lot, shuffle=True)` : des batchs, dans un ordre tiré à nouveau à chaque parcours

Attendu : `fabrique_dataset(X_train, y_train)` compte 391 batchs (`len`), et le premier, `next(iter(...))`, a des images de forme (128, 784).

```python
def fabrique_dataset(
  X: torch.Tensor, y: torch.Tensor, taille_lot: int = 128
) -> DataLoader:
  return DataLoader(TensorDataset(..., ...), batch_size=..., shuffle=...)


dataset = fabrique_dataset(X_train, y_train)
X_lot, y_lot = next(iter(dataset))
print(len(dataset), X_lot.shape, y_lot.shape)
```

### Solution

In [ ]:
def fabrique_dataset(
  X: torch.Tensor, y: torch.Tensor, taille_lot: int = 128
) -> DataLoader:
  # Les exemples sont mélangés à chaque epoch
  return DataLoader(TensorDataset(X, y), batch_size=taille_lot, shuffle=True)


dataset = fabrique_dataset(X_train, y_train)
X_lot, y_lot = next(iter(dataset))
print(len(dataset), X_lot.shape, y_lot.shape)

## Apprentissage

Voici une boucle d'apprentissage presque complète : elle parcourt les batchs à chaque epoch, puis mesure la perte et l'exactitude sur les ensembles d'entraînement et de validation, et trace leurs courbes à la fin.

*Implémentez la partie manquante : la mise à jour des paramètres du modèle pour un batch, par descente de gradient. Calculez la perte du batch, en lui ajoutant `regularisation(modele)` quand une régularisation est donnée (elle servira plus loin), remettez les gradients à zéro, calculez-les avec `backward`, puis retirez de chaque paramètre `taux_apprentissage` fois son gradient (slides « Différentiation automatique » et « Désactiver le suivi des gradients »).*

In [ ]:
FonctionPerte = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
FonctionRegularisation = Callable[[nn.Module], torch.Tensor]


def entraine(
  modele: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  nb_epoques: int = 10,
  taille_lot: int = 128,
  taux_apprentissage: float = 0.1,
  evalue_tous_les: int = 1,
  perte: FonctionPerte = entropie_croisee_categorielle,
  regularisation: FonctionRegularisation | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = modele(X_train)
    y_val_pred = modele(X_val)
    exactitudes = [exactitude(y_train, y_train_pred)]
    exactitudes_val = [exactitude(y_val, y_val_pred)]
    pertes = [float(perte(y_train_pred, y_train))]
    pertes_val = [float(perte(y_val_pred, y_val))]

  print(
    f"Métriques initiales : exactitude {exactitudes[-1]:.4f}, "
    f"exactitude_val {exactitudes_val[-1]:.4f}, "
    f"perte {pertes[-1]:.4f}, "
    f"perte_val {pertes_val[-1]:.4f}"
  )

  dataset_entrainement = fabrique_dataset(X_train, y_train, taille_lot)

  for e in range(nb_epoques):
    for X_lot, y_lot in dataset_entrainement:
      pass  # Votre code ici

    # Calcul et affichage des métriques toutes les `evalue_tous_les` epochs
    if (e + 1) % evalue_tous_les == 0:
      with torch.no_grad():
        y_train_pred = modele(X_train)
        y_val_pred = modele(X_val)
        exactitudes.append(exactitude(y_train, y_train_pred))
        pertes.append(float(perte(y_train_pred, y_train)))
        exactitudes_val.append(exactitude(y_val, y_val_pred))
        pertes_val.append(float(perte(y_val_pred, y_val)))
      print(
        f"Epoch {e + 1} : exactitude {exactitudes[-1]:.4f}, "
        f"exactitude_val {exactitudes_val[-1]:.4f}, "
        f"perte {pertes[-1]:.4f}, "
        f"perte_val {pertes_val[-1]:.4f}"
      )

  graduations_x = np.arange(0, nb_epoques + 1, evalue_tous_les)

  plt.plot(graduations_x, pertes, label="Entraînement")
  plt.plot(graduations_x, pertes_val, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(graduations_x, exactitudes, label="Entraînement")
  plt.plot(graduations_x, exactitudes_val, label="Validation")
  plt.title("Exactitude pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return exactitudes, exactitudes_val, pertes, pertes_val

### Conseils supplémentaires

- `perte(modele(X_lot), y_lot)` : la perte du batch ; on lui ajoute `regularisation(modele)` si `regularisation is not None`
- `modele.zero_grad()` : efface les gradients (les met à `None`), qui sinon s'accumulent d'un `backward` à l'autre
- `valeur_perte.backward()` : range le gradient de la perte dans l'attribut `.grad` de chaque paramètre
- `with torch.no_grad():` : la mise à jour ne doit pas entrer dans le graphe de calcul
- `parametre -= taux_apprentissage * parametre.grad` : la mise à jour, en place, pour chaque paramètre de `modele.parameters()`

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_lot, y_lot in dataset_entrainement:
  valeur_perte = ...
  if regularisation is not None:
    valeur_perte = valeur_perte + ...
  modele.zero_grad()
  valeur_perte.backward()
  with torch.no_grad():
    for parametre in modele.parameters():
      parametre -= ...
```

### Solution

In [ ]:
FonctionPerte = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
FonctionRegularisation = Callable[[nn.Module], torch.Tensor]


def entraine(
  modele: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  nb_epoques: int = 10,
  taille_lot: int = 128,
  taux_apprentissage: float = 0.1,
  evalue_tous_les: int = 1,
  perte: FonctionPerte = entropie_croisee_categorielle,
  regularisation: FonctionRegularisation | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = modele(X_train)
    y_val_pred = modele(X_val)
    exactitudes = [exactitude(y_train, y_train_pred)]
    exactitudes_val = [exactitude(y_val, y_val_pred)]
    pertes = [float(perte(y_train_pred, y_train))]
    pertes_val = [float(perte(y_val_pred, y_val))]

  print(
    f"Métriques initiales : exactitude {exactitudes[-1]:.4f}, "
    f"exactitude_val {exactitudes_val[-1]:.4f}, "
    f"perte {pertes[-1]:.4f}, "
    f"perte_val {pertes_val[-1]:.4f}"
  )

  dataset_entrainement = fabrique_dataset(X_train, y_train, taille_lot)

  for e in range(nb_epoques):
    for X_lot, y_lot in dataset_entrainement:
      # Calcul de la perte, avec la pénalité de régularisation s'il y en a une
      valeur_perte = perte(modele(X_lot), y_lot)
      if regularisation is not None:
        valeur_perte = valeur_perte + regularisation(modele)

      # Calcul automatique du gradient de la perte par rapport à chaque
      # paramètre : il est rangé dans l'attribut .grad du paramètre
      modele.zero_grad()
      valeur_perte.backward()

      # Mise à jour de chaque paramètre selon la règle :
      # nouvelle_valeur = ancienne_valeur - taux_apprentissage * gradient
      with torch.no_grad():
        for parametre in modele.parameters():
          parametre -= taux_apprentissage * parametre.grad

    # Calcul et affichage des métriques toutes les `evalue_tous_les` epochs
    if (e + 1) % evalue_tous_les == 0:
      with torch.no_grad():
        y_train_pred = modele(X_train)
        y_val_pred = modele(X_val)
        exactitudes.append(exactitude(y_train, y_train_pred))
        pertes.append(float(perte(y_train_pred, y_train)))
        exactitudes_val.append(exactitude(y_val, y_val_pred))
        pertes_val.append(float(perte(y_val_pred, y_val)))
      print(
        f"Epoch {e + 1} : exactitude {exactitudes[-1]:.4f}, "
        f"exactitude_val {exactitudes_val[-1]:.4f}, "
        f"perte {pertes[-1]:.4f}, "
        f"perte_val {pertes_val[-1]:.4f}"
      )

  graduations_x = np.arange(0, nb_epoques + 1, evalue_tous_les)

  plt.plot(graduations_x, pertes, label="Entraînement")
  plt.plot(graduations_x, pertes_val, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(graduations_x, exactitudes, label="Entraînement")
  plt.plot(graduations_x, exactitudes_val, label="Validation")
  plt.title("Exactitude pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return exactitudes, exactitudes_val, pertes, pertes_val

### Entraînement du réseau sans couche cachée

10 epochs, avec un learning rate de 0,5 :

In [ ]:
reseau_simple = ReseauSimple()
_ = entraine(reseau_simple, nb_epoques=10, taux_apprentissage=0.5)

*Quelle exactitude de validation atteint ce modèle ? Progresse-t-il encore après quelques epochs ?*

Votre réponse ici

#### Solution

Environ 92,5 % d'exactitude en validation. L'essentiel est appris dès la première epoch (environ 91,5 %) ; ensuite, le modèle ne gagne plus qu'un point en 9 epochs, en oscillant d'une epoch à l'autre. Les exactitudes d'entraînement et de validation restent égales : le modèle ne sur-apprend pas, il est trop simple. Une régression logistique sépare les classes par des frontières linéaires, ce qui plafonne vers 92 % sur MNIST.

## Avec un optimiseur PyTorch

La mise à jour écrite à la main ci-dessus (`parametre -= taux_apprentissage * parametre.grad`) est exactement ce que fait l'optimiseur `torch.optim.SGD`. En pratique, on écrit donc la boucle d'entraînement standard (slide « Boucle d'entraînement complète »), où un pas de l'optimiseur remplace la boucle sur les paramètres.

*Complétez la boucle de `entraine_avec_optimiseur`, qui fait un pas de `optimiseur` par batch : remise à zéro des gradients, perte du batch, calcul des gradients, mise à jour des paramètres. L'exercice suivant s'en sert pour comparer trois optimiseurs.*

In [ ]:
def entraine_avec_optimiseur(
  modele: nn.Module,
  optimiseur: torch.optim.Optimizer,
  nb_epoques: int = 5,
  taille_lot: int = 128,
) -> None:
  dataset_entrainement = fabrique_dataset(X_train, y_train, taille_lot)
  for e in range(nb_epoques):
    for X_lot, y_lot in dataset_entrainement:
      pass  # Votre code ici

    with torch.no_grad():
      exactitude_val = exactitude(y_val, modele(X_val))
    print(f"Epoch {e + 1} : exactitude_val {exactitude_val:.4f}")

### Conseils supplémentaires

- La perte se calcule comme dans `entraine`, avec `entropie_croisee_categorielle`, sans régularisation
- `optimiseur.zero_grad()`, `valeur_perte.backward()`, puis `optimiseur.step()` : un pas de l'optimiseur sur tous les paramètres qu'on lui a donnés à sa création

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_lot, y_lot in dataset_entrainement:
  optimiseur.zero_grad()
  valeur_perte = entropie_croisee_categorielle(..., ...)
  ...  # calcul des gradients
  ...  # mise à jour des paramètres
```

### Solution

In [ ]:
def entraine_avec_optimiseur(
  modele: nn.Module,
  optimiseur: torch.optim.Optimizer,
  nb_epoques: int = 5,
  taille_lot: int = 128,
) -> None:
  dataset_entrainement = fabrique_dataset(X_train, y_train, taille_lot)
  for e in range(nb_epoques):
    for X_lot, y_lot in dataset_entrainement:
      optimiseur.zero_grad()
      valeur_perte = entropie_croisee_categorielle(modele(X_lot), y_lot)
      valeur_perte.backward()
      optimiseur.step()

    with torch.no_grad():
      exactitude_val = exactitude(y_val, modele(X_val))
    print(f"Epoch {e + 1} : exactitude_val {exactitude_val:.4f}")

### Comparaison des optimiseurs

*Entraînez trois `ReseauSimple` neufs avec `entraine_avec_optimiseur`, 5 epochs chacun : le premier avec `torch.optim.SGD` et le learning rate de la boucle écrite à la main, 0,5 ; le deuxième avec SGD, un momentum de 0,9 et un learning rate dix fois plus petit, 0,05 ; le troisième avec `torch.optim.Adam` et son learning rate usuel, 10⁻³. Obtenez-vous avec SGD les performances de la boucle écrite à la main ? Que changent le momentum et Adam ?*

**Documentation utile**

- [`torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `torch.optim.SGD(modele.parameters(), lr=0.5)` : un optimiseur est lié aux paramètres d'un modèle ; un nouveau modèle demande un nouvel optimiseur
- `momentum=0.9` : un argument de `torch.optim.SGD`
- `torch.optim.Adam(modele.parameters(), lr=1e-3)` : Adam, avec son learning rate usuel

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print("SGD")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(
  reseau_simple, torch.optim.SGD(reseau_simple.parameters(), lr=...)
)

print("SGD avec momentum")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(
  reseau_simple, torch.optim.SGD(reseau_simple.parameters(), lr=..., momentum=...)
)

print("Adam")
...
```

#### Solution

In [ ]:
print("SGD")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(
  reseau_simple, torch.optim.SGD(reseau_simple.parameters(), lr=0.5)
)

print("SGD avec momentum")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(
  reseau_simple, torch.optim.SGD(reseau_simple.parameters(), lr=0.05, momentum=0.9)
)

print("Adam")
reseau_simple = ReseauSimple()
entraine_avec_optimiseur(
  reseau_simple, torch.optim.Adam(reseau_simple.parameters(), lr=1e-3)
)

Avec SGD et le même learning rate, on retrouve les performances de la boucle écrite à la main (environ 92 % en 5 epochs) : `optimiseur.step()` applique la même règle de mise à jour, et `optimiseur.zero_grad()` remplace `modele.zero_grad()`. Changer d'algorithme d'optimisation ne demande plus que de changer d'optimiseur.

Avec un momentum, SGD suit une moyenne glissante des gradients plutôt que le seul gradient du batch (`v = 0.9 * v + gradient`, puis `parametre -= lr * v`) : quand les gradients gardent la même direction, le pas vaut environ $1 / (1 - 0{,}9) = 10$ fois le learning rate. Avec un learning rate dix fois plus petit, il arrive donc au même résultat ; avec 0,5, ses pas seraient trop grands et l'exactitude oscillerait d'une epoch à l'autre. Sur ce modèle simple, il n'accélère pas l'apprentissage ; il sert surtout aux réseaux profonds, où il amortit les oscillations du gradient d'un batch à l'autre.

Adam adapte le pas de chaque paramètre à l'historique de ses gradients : avec un learning rate 500 fois plus petit, il arrive lui aussi au même résultat. Sur ce modèle simple, il n'apporte rien de plus ; son intérêt est que la valeur 10⁻³ convient à la plupart des réseaux, quand celle de SGD est à chercher pour chacun.

## Compiler le modèle avec `torch.compile`

Jusqu'ici, PyTorch exécute chaque opération dès que Python l'appelle (mode *eager*). `torch.compile` renvoie une version compilée du modèle : à son premier appel, PyTorch capture ses calculs dans un graphe, que TorchInductor compile en code optimisé pour le processeur ou le GPU, en fusionnant des opérations. Les appels suivants exécutent ce code compilé, tant que leurs entrées ressemblent à celles de la compilation (même forme, même `dtype`) : une entrée d'une autre forme déclenche une recompilation (slide « Compiler le modèle : `torch.compile` »).

La cellule suivante crée `modele`, un nouveau `ReseauSimple`, son optimiseur SGD, et `chronometre_epoque(reseau, nom)`, qui fait une epoch d'entraînement de `reseau` et affiche la durée de ses pas d'entraînement : leur total, celle du premier batch et celle du dernier. Elle chronomètre une epoch en mode eager.

*Compilez `modele` avec `torch.compile`, en gardant le nom `modele`, puis chronométrez deux epochs avec lui. Pourquoi le premier batch de la première epoch est-il si lent ? Et son dernier batch ? Une fois compilé, le modèle s'entraîne-t-il plus vite qu'en mode eager ?*

**Documentation utile**

- [`torch.compile`](https://docs.pytorch.org/docs/stable/generated/torch.compile.html)

In [ ]:
modele = ReseauSimple()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.5)
dataset_entrainement = fabrique_dataset(X_train, y_train)


def chronometre_epoque(reseau: nn.Module, nom: str) -> None:
  durees = []
  for X_lot, y_lot in dataset_entrainement:
    debut = time.perf_counter()
    optimiseur.zero_grad()
    valeur_perte = entropie_croisee_categorielle(reseau(X_lot), y_lot)
    valeur_perte.backward()
    optimiseur.step()
    durees.append(time.perf_counter() - debut)
  print(
    f"{nom} : {sum(durees):.2f} s, premier batch {durees[0]:.3f} s, "
    f"dernier batch {durees[-1]:.3f} s"
  )


chronometre_epoque(modele, "Eager")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `modele = torch.compile(modele)` : le modèle compilé, à appeler comme l'original ; il en partage les paramètres, et l'optimiseur créé avant la compilation reste valable
- `chronometre_epoque(modele, "Compilé, epoch 1")` : la durée d'une epoch avec le modèle compilé

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = torch.compile(...)
chronometre_epoque(..., "Compilé, epoch 1")
chronometre_epoque(..., "Compilé, epoch 2")
```

### Solution

In [ ]:
# Le modèle compilé partage les paramètres de l'original : l'optimiseur reste
# valable
modele = torch.compile(modele)
# Le premier batch compile le modèle, le dernier, plus petit, le recompile
chronometre_epoque(modele, "Compilé, epoch 1")
chronometre_epoque(modele, "Compilé, epoch 2")

Le premier batch prend plusieurs secondes, contre quelques millisecondes en mode eager : c'est la compilation. PyTorch capture les calculs du modèle et ceux de leurs gradients, puis TorchInductor génère du code C++ pour le processeur et le compile. Le dernier batch, de 80 exemples (50 000 = 390 $\times$ 128 + 80), a une autre forme que les autres : il déclenche une recompilation, d'une seconde ou deux. PyTorch compile alors une version où la taille du batch est libre : une troisième taille ne recompilerait plus. À la seconde epoch, plus rien ne se compile. `drop_last=True` dans le `DataLoader` éviterait la recompilation, au prix des 80 derniers exemples de chaque epoch.

Une fois compilé, le modèle ne s'entraîne pas plus vite sur le processeur : une epoch dure à peu près autant qu'en mode eager, parfois un peu plus. `ReseauSimple` ne fait qu'un produit matriciel et une addition, il y a peu à fusionner, et chaque appel du modèle compilé paie un surcoût fixe, comme la vérification que ses entrées correspondent à la version compilée. Les secondes de compilation ne sont jamais rattrapées. Le gain vient sur GPU, avec de plus gros modèles : les opérations fusionnées y évitent des allers-retours avec la mémoire. Sur processeur, on mesure donc avant de garder la compilation.

Le modèle compilé enveloppe l'original : les noms de son `state_dict` commencent par `_orig_mod.`. Pour sauvegarder les poids, on prend le `state_dict` de l'original, `modele._orig_mod.state_dict()`.

## Passage à des réseaux profonds

Pour passer à un perceptron multicouche, on crée une matrice de poids et un vecteur de biais par couche, avec `cree_poids_reseau_simple`. La fonction suivante les range dans une `nn.ModuleList`, un `nn.ParameterDict` `{"W": ..., "b": ...}` par couche : ces deux conteneurs permettent à `nn.Module` de trouver tous les paramètres (`modele.parameters()`).

In [ ]:
def cree_poids_mlp(
  nb_entrees: int, nb_sorties: int, tailles_couches_cachees: tuple[int, ...] = (512,)
) -> nn.ModuleList:
  couches = nn.ModuleList()
  # La taille d'entrée d'une couche est la taille de sortie de la précédente
  derniere_taille = nb_entrees
  for taille in [*tailles_couches_cachees, nb_sorties]:
    W, b = cree_poids_reseau_simple(derniere_taille, taille)
    couches.append(nn.ParameterDict({"W": W, "b": b}))
    derniere_taille = taille
  return couches

La sortie du modèle se calcule maintenant couche par couche : chaque couche prend en entrée le résultat de la précédente.

*Complétez la méthode `forward` de `MLP`. Chaque couche cachée calcule $\tanh(XW + b)$ avec `torch.tanh` ; la dernière renvoie les logits $XW + b$, sans fonction d'activation, comme `ReseauSimple`. Les paramètres de la couche `i` sont `self.couches[i]["W"]` et `self.couches[i]["b"]`.*

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    tailles_couches_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_mlp(nb_entrees, nb_sorties, tailles_couches_cachees)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Votre code ici

### Conseils supplémentaires

- `self.couches[:-1]` : les couches cachées ; `self.couches[-1]` : la couche de sortie
- `torch.tanh(X @ couche["W"] + couche["b"])` : une couche cachée

Attendu : `MLP()(X_train[:5])` a la forme (5, 10), et `MLP((256, 128))` a 6 paramètres (`len(list(....parameters()))`).

```python
class MLP(nn.Module):
  def __init__(
    self,
    tailles_couches_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_mlp(nb_entrees, nb_sorties, tailles_couches_cachees)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    for couche in self.couches[:-1]:
      X = torch.tanh(...)
    couche_sortie = self.couches[-1]
    return ... @ couche_sortie["W"] + ...
```

### Solution

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    tailles_couches_cachees: tuple[int, ...] = (512,),
    nb_entrees: int = 28**2,
    nb_sorties: int = 10,
  ) -> None:
    super().__init__()
    self.couches = cree_poids_mlp(nb_entrees, nb_sorties, tailles_couches_cachees)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Couches cachées : transformation affine, puis tanh
    for couche in self.couches[:-1]:
      X = torch.tanh(X @ couche["W"] + couche["b"])
    # Couche de sortie : les logits, sans activation
    couche_sortie = self.couches[-1]
    return X @ couche_sortie["W"] + couche_sortie["b"]


print(MLP()(X_train[:5]).shape, len(list(MLP((256, 128)).parameters())))

## Entraînement du réseau profond

Une couche cachée de 512 neurones, 10 epochs, un learning rate de 0,5 :

In [ ]:
mlp = MLP()
_ = entraine(mlp, nb_epoques=10, taux_apprentissage=0.5)

*Que constatez-vous en comparant ce réseau au réseau sans couche cachée ?*

Votre réponse ici

### Solution

Le réseau à une couche cachée fait nettement mieux que le réseau sans couche cachée : environ 97,5 % d'exactitude en validation, contre 92,5 %, soit une erreur divisée par 3. La couche cachée et sa tanh rendent le modèle non linéaire. L'exactitude d'entraînement approche 99 % et s'écarte de celle de validation : le réseau commence à sur-apprendre, mais sa perte de validation baisse encore. Avec un learning rate de 0,5, les courbes ne sont pas monotones : une epoch peut faire perdre un ou deux points, que les suivantes regagnent.

C'est en partie grâce à l'initialisation des poids, de variance $1 / n_\text{entrées}$. Pour vous en convaincre, retirez la division par `nb_entrees**0.5` dans `cree_poids_reseau_simple` et relancez cette cellule et l'entraînement : avec des poids de variance 1, les trois quarts des activations tanh sont saturées (au-delà de 0,99 en valeur absolue) dès le départ, les logits initiaux sont démesurés (une perte d'environ 31 au lieu de 2,3), et le réseau atteint 99,8 % sur l'entraînement mais seulement 93 % en validation. Ce grand écart ressemble à du sur-apprentissage, mais il vient d'une mauvaise initialisation (slide « Initialisation des poids »).

## Régularisation

Le perceptron précédent sur-apprend peu : 50 000 exemples suffisent à ses 400 000 poids. Le sur-apprentissage apparaît quand le réseau est grand par rapport aux données : la cellule qui suit la solution entraîne le même perceptron sur les 1 000 premiers exemples seulement, 100 epochs, sans puis avec régularisation.

La régularisation L2 ajoute à la perte une pénalité, $\lambda \sum w^2$ : $\lambda$ fois la somme des carrés des poids, qui pousse les poids vers 0 (slide « Par pénalisation des poids »). On ne pénalise que les poids `W`, pas les biais : ils ne multiplient pas l'entrée, et les contraindre n'aide pas à généraliser.

*Complétez la fonction interne de `l2`, qui renvoie la pénalité d'un `MLP` : la somme des carrés des matrices `W` de toutes ses couches (`modele.couches`), multipliée par `coefficient_lambda`.*

**Documentation utile**

- [`torch.sum`](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

In [ ]:
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: MLP) -> torch.Tensor:
    pass  # Votre code ici

  return penalite

### Conseils supplémentaires

- `torch.sum(couche["W"] ** 2)` : la somme des carrés d'une matrice
- `sum(...)` de Python, sur toutes les couches de `modele.couches`, pour les additionner

Attendu : pour un `MLP()` neuf, `l2(1.0)(MLP())` vaut environ 522 (512 pour la première couche, 10 pour la seconde : chacune contribue à peu près son nombre de colonnes, puisque chaque poids a une variance de $1 / n_\text{entrées}$).

```python
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: MLP) -> torch.Tensor:
    norme_poids = sum(torch.sum(...) for couche in modele.couches)
    return ... * norme_poids

  return penalite
```

### Solution

In [ ]:
def l2(coefficient_lambda: float) -> FonctionRegularisation:
  def penalite(modele: MLP) -> torch.Tensor:
    # Les poids W seulement, pas les biais
    norme_poids = sum(torch.sum(couche["W"] ** 2) for couche in modele.couches)
    return coefficient_lambda * norme_poids

  return penalite


# .item() : la valeur, en nombre Python, hors du graphe d'autograd
print(l2(1.0)(MLP()).item())

### Sur-apprentissage sur 1 000 exemples

Le même perceptron, avec le même learning rate de 0,5, sur les 1 000 premiers exemples d'entraînement, sans régularisation puis avec une pénalité L2 de $\lambda$ = 10⁻³ ; la validation reste faite sur les 10 000 exemples mis de côté.

In [ ]:
X_petit, y_petit = X_train[:1000], y_train[:1000]

print("Sans régularisation")
_ = entraine(
  MLP(),
  X_train=X_petit,
  y_train=y_petit,
  nb_epoques=100,
  taux_apprentissage=0.5,
  evalue_tous_les=10,
)

print("Avec une pénalité L2")
_ = entraine(
  MLP(),
  X_train=X_petit,
  y_train=y_petit,
  nb_epoques=100,
  taux_apprentissage=0.5,
  evalue_tous_les=10,
  regularisation=l2(1e-3),
)

*Comparez les deux entraînements : l'écart entre entraînement et validation, la perte de validation au fil des epochs, l'exactitude de validation finale.*

Votre réponse ici

#### Solution

Sans régularisation, le réseau apprend par cœur les 1 000 exemples : 100 % d'exactitude d'entraînement dès la trentième epoch, une perte d'entraînement qui tend vers 0, et une exactitude de validation bloquée vers 87 %. La perte de validation est au plus bas dans les 10 premières epochs, puis remonte sans cesse, de 0,44 à 0,60 : le réseau devient de plus en plus sûr de ses réponses, y compris des fausses. C'est le sur-apprentissage.

Avec la pénalité L2, la perte d'entraînement reste au-dessus de 0,027 (les poids ne peuvent plus grandir librement), et la perte de validation remonte beaucoup moins : elle se stabilise vers 0,47. L'exactitude de validation, elle, ne change pas : sur si peu de données, la régularisation limite surtout l'excès de confiance du modèle. Rien ne remplace davantage de données : le même réseau atteint 97,5 % avec les 50 000 exemples.

## Suivre l'entraînement avec TensorBoard

Les courbes que trace `entraine` disparaissent avec le notebook, et se comparent mal d'un entraînement à l'autre. TensorBoard affiche des journaux, écrits pendant l'entraînement :

- `SummaryWriter(dossier)`, de `torch.utils.tensorboard`, crée un journal dans `dossier` ;
- `journal.add_scalar(nom, valeur, epoque)` ajoute un point à la courbe `nom` ;
- `journal.close()` écrit sur le disque ce qui reste en mémoire.

Un dossier par essai : TensorBoard superpose leurs courbes, ce qui permet de comparer des hyper-paramètres sur l'ensemble de validation, jamais sur le test. Comparer quelques valeurs d'un hyper-paramètre, c'est une recherche en grille (slide « Recherche d'hyper-paramètres : grille ou aléatoire ? »).

*Complétez `entraine_avec_journal` : à chaque epoch, faites un pas de `optimiseur` par batch, comme dans `entraine_avec_optimiseur`, puis écrivez dans le journal la perte moyenne de l'epoch (`perte`) et l'exactitude de validation (`exactitude_val`). La fonction renvoie la dernière exactitude de validation. La cellule qui suit la solution entraîne un MLP par SGD avec trois learning rates, chacun dans son dossier de `journaux/`, puis ouvre TensorBoard.*

**Documentation utile**

- [`SummaryWriter`](https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter)
- [`add_scalar`](https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar)

In [ ]:
def entraine_avec_journal(
  modele: nn.Module,
  optimiseur: torch.optim.Optimizer,
  dossier: str,
  nb_epoques: int = 5,
) -> float:
  pass  # Votre code ici

### Conseils supplémentaires

- `journal = SummaryWriter(dossier)` au début de la fonction, `journal.close()` à la fin
- La boucle sur les batchs de `entraine_avec_optimiseur` ; `valeur_perte.item()`, ajouté à une liste `pertes`, et `np.mean(pertes)` : la perte moyenne de l'epoch
- `exactitude(y_val, modele(X_val))`, sous `torch.no_grad()` : l'exactitude de validation
- `journal.add_scalar("perte", ..., e + 1)` : l'abscisse du point, en epochs

```python
def entraine_avec_journal(
  modele: nn.Module,
  optimiseur: torch.optim.Optimizer,
  dossier: str,
  nb_epoques: int = 5,
) -> float:
  journal = SummaryWriter(...)
  dataset_entrainement = fabrique_dataset(X_train, y_train)
  for e in range(nb_epoques):
    pertes = []
    for X_lot, y_lot in dataset_entrainement:
      ...  # un pas de l'optimiseur, comme dans entraine_avec_optimiseur
      pertes.append(valeur_perte.item())
    with torch.no_grad():
      exactitude_val = exactitude(..., ...)
    journal.add_scalar("perte", ..., e + 1)
    journal.add_scalar(..., ..., ...)
  journal.close()
  return exactitude_val
```

### Solution

In [ ]:
def entraine_avec_journal(
  modele: nn.Module,
  optimiseur: torch.optim.Optimizer,
  dossier: str,
  nb_epoques: int = 5,
) -> float:
  journal = SummaryWriter(dossier)
  dataset_entrainement = fabrique_dataset(X_train, y_train)
  for e in range(nb_epoques):
    pertes = []
    for X_lot, y_lot in dataset_entrainement:
      optimiseur.zero_grad()
      valeur_perte = entropie_croisee_categorielle(modele(X_lot), y_lot)
      valeur_perte.backward()
      optimiseur.step()
      pertes.append(valeur_perte.item())
    with torch.no_grad():
      exactitude_val = exactitude(y_val, modele(X_val))
    # Un point par courbe et par epoch
    journal.add_scalar("perte", np.mean(pertes), e + 1)
    journal.add_scalar("exactitude_val", exactitude_val, e + 1)
    print(f"Epoch {e + 1} : exactitude_val {exactitude_val:.4f}")
  journal.close()
  return exactitude_val

### Comparaison de trois learning rates

Un MLP par learning rate, 5 epochs chacun, journalisés dans `journaux/sgd_<learning rate>` :

In [ ]:
mlps, exactitudes_val = {}, {}
for taux in (0.01, 0.1, 0.5):
  print(f"Learning rate {taux}")
  mlps[taux] = MLP()
  optimiseur = torch.optim.SGD(mlps[taux].parameters(), lr=taux)
  exactitudes_val[taux] = entraine_avec_journal(
    mlps[taux], optimiseur, f"journaux/sgd_{taux}"
  )

In [ ]:
%load_ext tensorboard
%tensorboard --logdir journaux

*Dans TensorBoard, quel learning rate donne la meilleure exactitude de validation ? Que montrent les courbes de perte ?*

Votre réponse ici

#### Solution

0,5 : environ 96,5 à 97 % d'exactitude de validation en 5 epochs, contre 93,5 % pour 0,1 et 90 % pour 0,01 (courbe `exactitude_val`). Les courbes de `perte` montrent pourquoi : plus le learning rate est petit, plus la perte d'entraînement descend lentement. À 0,01, le réseau est loin d'avoir fini d'apprendre en 5 epochs : comparer des essais aussi courts favorise ce qui apprend vite, pas forcément ce qui finit le mieux.

TensorBoard relit les journaux à mesure qu'ils s'écrivent : on peut suivre un entraînement en cours, avec un peu de retard, `SummaryWriter` n'écrivant sur le disque que toutes les deux minutes ou tous les dix points (`flush_secs` et `max_queue`). Relancer la cellule ajoute de nouveaux journaux aux anciens, dans les mêmes dossiers : supprimez `journaux/` pour repartir de zéro.

## Évaluation finale sur l'ensemble de test

L'ensemble de test n'a servi à rien jusqu'ici : ni à l'entraînement, ni au choix du learning rate. C'est ce qui rend son évaluation fiable.

*Prenez `mlp`, le MLP entraîné avec le learning rate de la meilleure exactitude de validation, et calculez son exactitude sur `X_test`. Est-elle proche de son exactitude de validation ?*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `max(exactitudes_val, key=exactitudes_val.get)` : la clé de la plus grande valeur du dictionnaire, ici le meilleur learning rate
- `mlps[meilleur_taux]` : le MLP entraîné avec lui
- `exactitude(y_test, mlp(X_test))`, sous `torch.no_grad()` : l'exactitude de test

```python
meilleur_taux = max(..., key=...)
mlp = mlps[...]
with torch.no_grad():
  print(f"Exactitude sur l'ensemble de test : {exactitude(..., ...):.4f}")
```

### Solution

In [ ]:
# Le learning rate de la meilleure exactitude de validation, et son MLP
meilleur_taux = max(exactitudes_val, key=exactitudes_val.get)
mlp = mlps[meilleur_taux]
print(f"Learning rate retenu : {meilleur_taux}")
print(f"Exactitude sur l'ensemble de validation : {exactitudes_val[meilleur_taux]:.4f}")
with torch.no_grad():
  print(f"Exactitude sur l'ensemble de test : {exactitude(y_test, mlp(X_test)):.4f}")

L'exactitude de test est proche de l'exactitude de validation, à un demi-point près (par exemple 96,6 % contre 97,1 %). La validation a servi à choisir le learning rate : son exactitude est en principe légèrement optimiste, d'autant plus que les essais comparés sont nombreux. L'ensemble de test, qui n'a servi à aucun choix, donne une estimation fiable des performances du modèle sur de nouvelles données.

Avec un learning rate de 0,5, la dernière epoch n'est pas toujours la meilleure : l'exactitude de validation peut redescendre d'une epoch à l'autre, de quelques dixièmes de point sur ces 5 epochs, d'un ou deux points sur les 10 de « Entraînement du réseau profond ». Garder les poids de la meilleure epoch de validation, c'est l'arrêt précoce (slide « Par early stopping »).

## Sauvegarder et recharger le modèle

Un entraînement coûte cher : on sauvegarde le modèle entraîné pour le réutiliser sans le réentraîner. On sauvegarde son `state_dict`, le dictionnaire de ses paramètres rangés par noms, plutôt que l'objet Python lui-même (slide « Sauvegarder et recharger un modèle »).

*Sauvegardez le `state_dict` de `mlp` dans le fichier `mlp.pt`. Créez `modele_restaure`, un nouveau `MLP` de même architecture, et affichez son exactitude de test avant et après y avoir chargé ces poids.*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `torch.save(mlp.state_dict(), "mlp.pt")` : la sauvegarde ; les clés du `state_dict` sont les chemins d'attributs des paramètres (`couches.0.W`, `couches.0.b`…)
- `modele_restaure` doit avoir la même architecture que `mlp`, un `MLP()` à ses tailles par défaut : mêmes noms de paramètres, mêmes formes
- `torch.load("mlp.pt", map_location=device)` : relit le dictionnaire, ses tenseurs placés sur `device` ; `modele_restaure.load_state_dict(...)` copie ses valeurs dans les paramètres de `modele_restaure`
- `exactitude(y_test, modele_restaure(X_test))`, sous `torch.no_grad()` : l'exactitude de test

Attendu : environ 0,1 avant le chargement, l'exactitude de `mlp` après, à l'identique.

```python
torch.save(..., "mlp.pt")
modele_restaure = ...
with torch.no_grad():
  print(exactitude(y_test, modele_restaure(X_test)))
modele_restaure.load_state_dict(torch.load(..., map_location=...))
with torch.no_grad():
  print(exactitude(..., ...))
```

### Solution

In [ ]:
torch.save(mlp.state_dict(), "mlp.pt")
# Les clés : le chemin d'attributs de chaque paramètre
print(list(mlp.state_dict().keys()))

# Nouveaux poids aléatoires, même architecture que mlp
modele_restaure = MLP()
with torch.no_grad():
  print(f"Avant chargement : {exactitude(y_test, modele_restaure(X_test)):.4f}")
modele_restaure.load_state_dict(torch.load("mlp.pt", map_location=device))
with torch.no_grad():
  print(f"Après chargement : {exactitude(y_test, modele_restaure(X_test)):.4f}")
  print(f"Modèle sauvegardé : {exactitude(y_test, mlp(X_test)):.4f}")

Avant le chargement, `modele_restaure` répond presque au hasard (de l'ordre de 10 %, ses poids sont aléatoires). Après, il retrouve à l'identique l'exactitude de `mlp` : `load_state_dict` a copié chaque tenseur dans le paramètre de même nom (`couches.0.W`, `couches.0.b`, `couches.1.W`, `couches.1.b`). D'où l'importance d'une architecture identique : un `MLP` d'une autre taille aurait des paramètres de formes différentes, et le chargement échouerait. `map_location` place les tenseurs lus sur `device` : sans lui, des poids sauvegardés sur un GPU ne se chargent pas sur une machine qui n'en a pas.

Le fichier ne contient que des valeurs, pas le code du modèle : pour le recharger, il faut la classe `MLP`. Notre `MLP` n'a ni dropout ni normalisation de batch ; pour un modèle qui en a, on appelle `modele_restaure.eval()` avant de prédire. Et pour reprendre un entraînement, on sauvegarde aussi `optimiseur.state_dict()` et le numéro d'epoch.